# Qwen3-VL Domain Base — Quantitative Eval (post-overnight-train)

**Why this exists:** the overnight run's smoke test (n=1/task) showed correct templates but
wrong content (count 0 vs 24, degenerate OCR). Masking was verified correct — the training was
real, so the question is *how much* visual grounding the adapter actually has. This notebook
answers it with ~50 examples per task, scored, base-vs-adapter side by side.

**Metrics per task:**
- `count`: MAE + % exactly right + % answering "0" (tests the learned-the-prior hypothesis)
- `typed_summary`: class-set F1 (predicted class names vs true set)
- `relation`: accuracy vs the 0.5 coin-flip baseline
- `ocr`: tag recall (share of true Tesseract tags reproduced) + degeneration rate

**Both models run on the SAME eval examples:** the raw base (no adapter) is the control —
if the adapter doesn't beat the raw base, the overnight run added nothing.

**Eval pool is built with a different seed / disjoint slice from training** where possible
(Kaggle: images beyond the first 1000; PID2Graph: patches not sampled by the training seed;
Gupta: same 72 sheets — unavoidable, only 72 exist — but different tiles are not guaranteed,
so Gupta numbers are best read as "training-set fit", which is still diagnostic here: the
smoke test failed ON training data).

## 1. Config

In [ ]:
HF_TOKEN = "paste-your-hf-token-here"

DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
ADAPTER_PATH_IN_REPO = "latest"      # or "epoch_0"/"epoch_1"/"epoch_2" to eval a specific snapshot
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
N_PER_TASK = 50

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here", "Paste your HF token"

## 2. Install + GPU check

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q peft pytesseract huggingface_hub
!pip uninstall -y torchao -q

import torch
assert torch.cuda.is_available(), "Needs a GPU (A100 is fine for eval)"
print(torch.cuda.get_device_name(0))

## 3. Data from HF → local disk (same as training notebook)

In [ ]:
import zipfile, time
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

t0 = time.time()
for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("kaggle_pid_symbols/kaggle_pid_symbols.zip", DATA / "kaggle"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} already extracted, skipping")
        continue
    zp = hf_hub_download(repo_id=DATA_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted {fname} ({time.time()-t0:.0f}s)")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
KAGGLE_IMAGES = DATA / "kaggle" / "archive" / "images (3)"
KAGGLE_LABELS = DATA / "kaggle" / "archive" / "labels (2)"
PID2GRAPH_OPEN100 = DATA / "pid2graph" / "PID2Graph" / "Patched" / "PID2Graph OPEN100"
for p in [GUPTA_RAW, KAGGLE_IMAGES, KAGGLE_LABELS, PID2GRAPH_OPEN100]:
    assert p.exists(), p
print("data ready")

## 4. Build eval pools (~50/task)

Same builders as training, but sliced to minimize train overlap: Kaggle takes images AFTER the
first 1000 (training used the first 1000); relation pairs use seed=999 (training used seed=0,
so both the patch subset and the sampled pairs differ); Gupta reuses the 72 train sheets
(unavoidable) with tiles sampled at seed=999.

In [ ]:
import json, random
import xml.etree.ElementTree as ET
import pytesseract
from PIL import Image
from collections import Counter

Image.MAX_IMAGE_PIXELS = None
TILE_SIZE, OVERLAP = 1024, 205
MIN_TILE_SIDE = 64   # skip degenerate edge slivers (the aspect-ratio crashes in training)

CLASS_NAMES = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            if (x1 - x0) >= MIN_TILE_SIDE and (y1 - y0) >= MIN_TILE_SIDE:
                tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return parts[0], [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def boxes_intersect(a, b):
    return a[0] < b[2] and a[2] > b[0] and a[1] < b[3] and a[3] > b[1]

def ocr_tags(tile_img, min_len=2):
    tokens = [t.strip() for t in pytesseract.image_to_string(tile_img).split() if len(t.strip()) >= min_len]
    return sorted(set(t for t in tokens if any(c.isdigit() for c in t) or "-" in t))

OCR_PROMPT = "List every text tag or label you can read in this P&ID tile."
COUNT_PROMPT = "How many distinct symbols (valves, instruments, fittings, etc.) are visible in this P&ID tile?"
TYPED_SUMMARY_PROMPT = "What types of symbols are visible in this P&ID tile, and how many of each?"
RELATION_PROMPT_TEMPLATE = (
    "In this P&ID crop there is a symbol at [{ax0}, {ay0}, {ax1}, {ay1}] and another at "
    "[{bx0}, {by0}, {bx1}, {by1}] (pixel coordinates, top-left origin). "
    "Are these two symbols directly connected to each other?"
)
MAX_PAIR_SPAN_PX = 1400
CROP_MARGIN = 80

rng = random.Random(999)   # training used seed 0

# --- count + ocr eval (Gupta) ---
count_pool, ocr_pool = [], []
sheet_paths = sorted((GUPTA_RAW / "sheets" / "train").glob("*.*"))
rng.shuffle(sheet_paths)
for sheet_path in sheet_paths:
    if len(count_pool) >= N_PER_TASK and len(ocr_pool) >= N_PER_TASK:
        break
    label_path = GUPTA_RAW / "labels" / "train" / f"{sheet_path.stem}.txt"
    img = Image.open(sheet_path).convert("RGB")
    W, H = img.size
    orig_boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                orig_boxes.append(yolo_line_to_xyxy(line, W, H)[1])
    tiles = compute_tile_grid(W, H)
    rng.shuffle(tiles)
    for tbox in tiles[:4]:
        n_boxes = sum(1 for b in orig_boxes if boxes_intersect(b, tbox))
        if len(count_pool) < N_PER_TASK:
            count_pool.append({"image_path": str(sheet_path), "crop": list(tbox),
                               "prompt": COUNT_PROMPT, "true_count": n_boxes})
        if len(ocr_pool) < N_PER_TASK:
            tags = ocr_tags(img.crop(tbox))
            if tags:
                ocr_pool.append({"image_path": str(sheet_path), "crop": list(tbox),
                                 "prompt": OCR_PROMPT, "true_tags": tags})

# --- typed_summary eval (Kaggle, images AFTER the first 1000 = disjoint from training) ---
typed_pool = []
for lbl in sorted(KAGGLE_LABELS.glob("*.txt"))[1000:1000 + N_PER_TASK * 3]:
    if len(typed_pool) >= N_PER_TASK:
        break
    lines = [l for l in lbl.read_text().splitlines() if l.strip()]
    img_path = KAGGLE_IMAGES / f"{lbl.stem}.jpg"
    if not lines or not img_path.exists():
        continue
    true_classes = {CLASS_NAMES.get(l.split()[0], "?") for l in lines}
    typed_pool.append({"image_path": str(img_path), "crop": None,
                       "prompt": TYPED_SUMMARY_PROMPT, "true_classes": true_classes})

# --- relation eval (PID2Graph, seed-disjoint pair sampling) ---
def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

relation_pool = []
gml_files = sorted(PID2GRAPH_OPEN100.rglob("*.graphml"))
rng.shuffle(gml_files)
for gml in gml_files:
    if len(relation_pool) >= N_PER_TASK:
        break
    png = gml.with_suffix(".png")
    if not png.exists():
        continue
    try:
        nodes, edges = parse_graphml(gml)
    except ET.ParseError:
        continue
    if len(nodes) < 4 or not edges:
        continue
    node_ids = list(nodes)
    want_connected = len(relation_pool) % 2 == 0   # alternate -> balanced 50/50
    pair = None
    if want_connected:
        pool = [tuple(e) for e in edges]
        rng.shuffle(pool)
        pair = pool[0] if pool else None
    else:
        for _ in range(50):
            a, b = rng.sample(node_ids, 2)
            if frozenset((a, b)) not in edges:
                pair = (a, b)
                break
    if pair is None:
        continue
    a, b = pair
    ba, bb = nodes[a], nodes[b]
    ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
    ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
    if ux1 - ux0 > MAX_PAIR_SPAN_PX or uy1 - uy0 > MAX_PAIR_SPAN_PX:
        continue
    cx0, cy0 = max(0, int(ux0 - CROP_MARGIN)), max(0, int(uy0 - CROP_MARGIN))
    cx1, cy1 = int(ux1 + CROP_MARGIN), int(uy1 + CROP_MARGIN)
    prompt = RELATION_PROMPT_TEMPLATE.format(
        ax0=int(ba[0] - cx0), ay0=int(ba[1] - cy0), ax1=int(ba[2] - cx0), ay1=int(ba[3] - cy0),
        bx0=int(bb[0] - cx0), by0=int(bb[1] - cy0), bx1=int(bb[2] - cx0), by1=int(bb[3] - cy0))
    relation_pool.append({"image_path": str(png), "crop": [cx0, cy0, cx1, cy1],
                          "prompt": prompt, "connected": want_connected})

print(f"eval pools: count={len(count_pool)} ocr={len(ocr_pool)} "
      f"typed={len(typed_pool)} relation={len(relation_pool)}")
print("count distribution (true):", sorted(Counter(e['true_count'] for e in count_pool).items()))

## 5. Load base model + generation helper

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
model.eval()
print("base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def generate(ex, max_new_tokens=96):
    img = Image.open(ex["image_path"]).convert("RGB")
    if ex.get("crop"):
        img = img.crop(ex["crop"])
    messages = [{"role": "user", "content": [{"type": "image", "image": img},
                                             {"type": "text", "text": ex["prompt"]}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 6. Scoring + eval runner (used for both base and adapter)

In [ ]:
import re

def score_count(pred_text, true_count):
    m = re.search(r"\b(\d+)\b", pred_text)
    pred = int(m.group(1)) if m else None
    return {"pred": pred, "true": true_count,
            "abs_err": abs(pred - true_count) if pred is not None else None,
            "exact": pred == true_count, "said_zero": pred == 0}

def score_typed(pred_text, true_classes):
    known = set(CLASS_NAMES.values()) - {"Not_used"}
    pred_classes = {name for name in known if name.lower() in pred_text.lower()}
    tp = len(pred_classes & true_classes)
    prec = tp / len(pred_classes) if pred_classes else 0.0
    rec = tp / len(true_classes) if true_classes else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"f1": f1, "precision": prec, "recall": rec}

def score_relation(pred_text, connected):
    t = pred_text.lower()
    yes = t.startswith("yes") or ("yes" in t[:40] and "no" not in t[:15])
    no = t.startswith("no")
    if not yes and not no:
        return {"correct": None}
    return {"correct": (yes and connected) or (no and not connected)}

def score_ocr(pred_text, true_tags):
    found = sum(1 for tag in true_tags if tag in pred_text)
    words = pred_text.split()
    degenerate = len(words) > 8 and len(set(words)) / len(words) < 0.3
    return {"tag_recall": found / len(true_tags), "degenerate": degenerate}

def run_eval(label):
    results = {}
    t0 = time.time()
    # count
    rows = [score_count(generate(ex), ex["true_count"]) for ex in count_pool]
    errs = [r["abs_err"] for r in rows if r["abs_err"] is not None]
    results["count"] = {
        "n": len(rows), "MAE": sum(errs) / len(errs) if errs else None,
        "exact_%": 100 * sum(r["exact"] for r in rows) / len(rows),
        "answered_zero_%": 100 * sum(r["said_zero"] for r in rows) / len(rows),
        "true_zero_%": 100 * sum(1 for e in count_pool if e["true_count"] == 0) / len(count_pool),
        "unparseable": sum(1 for r in rows if r["pred"] is None)}
    print(f"[{label}] count done {time.time()-t0:.0f}s")
    # typed
    rows = [score_typed(generate(ex), ex["true_classes"]) for ex in typed_pool]
    results["typed_summary"] = {"n": len(rows),
        "class_F1": sum(r["f1"] for r in rows) / len(rows),
        "precision": sum(r["precision"] for r in rows) / len(rows),
        "recall": sum(r["recall"] for r in rows) / len(rows)}
    print(f"[{label}] typed done {time.time()-t0:.0f}s")
    # relation
    rows = [score_relation(generate(ex, max_new_tokens=32), ex["connected"]) for ex in relation_pool]
    decided = [r for r in rows if r["correct"] is not None]
    results["relation"] = {"n": len(rows),
        "accuracy_%": 100 * sum(r["correct"] for r in decided) / len(decided) if decided else None,
        "undecided": len(rows) - len(decided), "baseline_%": 50.0}
    print(f"[{label}] relation done {time.time()-t0:.0f}s")
    # ocr
    rows = [score_ocr(generate(ex, max_new_tokens=128), ex["true_tags"]) for ex in ocr_pool]
    results["ocr"] = {"n": len(rows),
        "tag_recall_%": 100 * sum(r["tag_recall"] for r in rows) / len(rows),
        "degenerate_%": 100 * sum(r["degenerate"] for r in rows) / len(rows)}
    print(f"[{label}] ocr done {time.time()-t0:.0f}s")
    return results

def show(results, label):
    print(f"\n===== {label} =====")
    for task, m in results.items():
        print(f"  {task}:")
        for k, v in m.items():
            print(f"    {k:18s} {v if not isinstance(v, float) else round(v, 3)}")

## 7. Eval 1 — RAW BASE (control, no adapter)

In [ ]:
base_results = run_eval("base")
show(base_results, "RAW BASE (no adapter)")

## 8. Eval 2 — TRAINED ADAPTER (pulled from HF)

In [ ]:
from huggingface_hub import snapshot_download

CKPT_LOCAL = Path("/content/eval_ckpt")
snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                  allow_patterns=[f"{ADAPTER_PATH_IN_REPO}/*"], local_dir=str(CKPT_LOCAL))
adapter_dir = CKPT_LOCAL / ADAPTER_PATH_IN_REPO
assert (adapter_dir / "adapter_model.safetensors").exists(), f"no adapter found at {adapter_dir}"

from peft import PeftModel
model = PeftModel.from_pretrained(model, str(adapter_dir))
model.eval()
print(f"adapter loaded from {CKPT_REPO}/{ADAPTER_PATH_IN_REPO}")

adapter_results = run_eval("adapter")
show(adapter_results, f"TRAINED ADAPTER ({ADAPTER_PATH_IN_REPO})")

## 9. Side-by-side verdict

In [ ]:
print(f"{'metric':38s} {'base':>10s} {'adapter':>10s}")
print("-" * 60)
for task in base_results:
    for k in base_results[task]:
        b, a = base_results[task][k], adapter_results[task][k]
        fmt = lambda v: f"{v:.2f}" if isinstance(v, float) else str(v)
        print(f"{task + '.' + k:38s} {fmt(b):>10s} {fmt(a):>10s}")

print("""
How to read this:
- adapter no better than base anywhere -> overnight run added ~nothing; retrain with
  rebalanced/diversified data before spending more compute downstream.
- adapter beats base on relation/typed but not count/ocr -> keep the adapter for the
  connectivity/typing stages; fix the count/ocr task design and continue training FROM
  this checkpoint.
- count.answered_zero_% >> count.true_zero_% (adapter only) -> confirms it learned the
  '0 is the safe answer' prior from the skewed training pool.
- ocr.degenerate_% high -> the Tesseract task taught format, not reading; drop or de-noise
  it next round.""")